In [17]:
"""
Calcula las métricas de concordancia entre los 5 modelos a partir de
resultados_clustering_v2.csv (word_normalized, modelo, n_ocurrencias, k_clusters,
pct_ruido, presenta_multiples_clusters).

IMPORTANTE — alcance de lo que estas métricas pueden afirmar: `k_clusters` y
`presenta_multiples_clusters` describen la ESTRUCTURA de clusters observada en
los embeddings de cada modelo, no polisemia léxica validada (ver limitación
documentada en clusterizar_sentidos.py y en el protocolo, sección 9). Por lo
tanto, el Kappa/Jaccard/Fleiss calculados aquí miden CONCORDANCIA EN
ESTRUCTURA DE EMBEDDINGS entre modelos, no concordancia en detección de
polisemia real.

Uso:
    python calcular_concordancia.py
"""

import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

RESULTADOS_CSV = "resultados_clustering_v2.csv"
MODELOS = ["mex_state", "mex_large", "beto", "bertin", "robertuito"]
N_COMPARACIONES = len(MODELOS) * (len(MODELOS) - 1) // 2  # 10
ALPHA_BONFERRONI = 0.05 / N_COMPARACIONES  # 0.005 (sección 6 del protocolo)
N_BOOTSTRAP = 1000
UMBRAL_CONSENSO = 3  # de 5 modelos, para definir "palabra de consenso" (sección 6/8)

# Umbral de silhouette del análisis de Youden sobre palabras de control
# (sensibilidad 0.537, especificidad 0.678, J=0.215 -- sección 9 del protocolo).
# La columna presenta_multiples_clusters guardada en el CSV NO incorpora este
# umbral (solo usa k>=2 y ruido<=50%, que por sí solo no discrimina -- por eso
# daba 100% en todos los modelos). Se recalcula aquí antes de cualquier análisis.
UMBRAL_SILHOUETTE = 0.68

rng = np.random.default_rng(42)


def bootstrap_kappa_ci(y1, y2, n_boot=N_BOOTSTRAP, alpha=ALPHA_BONFERRONI):
    n = len(y1)
    idx = np.arange(n)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(idx, size=n, replace=True)
        # kappa indefinido si todo el sample cae en una sola clase en ambos raters
        if len(set(y1[sample])) < 2 and len(set(y2[sample])) < 2:
            continue
        boots.append(cohen_kappa_score(y1[sample], y2[sample]))
    if not boots:
        return np.nan, np.nan
    lower = np.percentile(boots, 100 * alpha / 2)
    upper = np.percentile(boots, 100 * (1 - alpha / 2))
    return lower, upper


def fleiss_kappa(tabla_bool):
    """tabla_bool: DataFrame (palabra x modelo) booleano, sin NaN."""
    n_items, n_raters = tabla_bool.shape
    votos_si = tabla_bool.sum(axis=1).values.astype(float)
    votos_no = n_raters - votos_si
    P_i = (votos_si**2 + votos_no**2 - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()
    p_si = votos_si.sum() / (n_items * n_raters)
    p_no = 1 - p_si
    P_e = p_si**2 + p_no**2
    return (P_bar - P_e) / (1 - P_e)



df = pd.read_csv(RESULTADOS_CSV)

# Recalcular presenta_multiples_clusters incorporando el umbral de silhouette
# (la columna guardada por clusterizar_sentidos.py solo usa k>=2 y ruido<=50%,
# que por sí solo no discrimina -- ver limitación en el protocolo, sección 9).
def _decidir(row):
    if pd.isna(row["k_clusters"]) or pd.isna(row["pct_ruido"]):
        return np.nan  # no evaluada (pocas ocurrencias) o excluida por stopword
    if row["pct_ruido"] > 50:
        return np.nan  # "no determinado", ajuste de mala calidad
    if row["k_clusters"] < 2:
        return False
    if pd.isna(row["silhouette"]) or row["silhouette"] < UMBRAL_SILHOUETTE:
        return False  # split encontrado, pero no pasa el filtro de calidad de separación
    return True    
df["presenta_multiples_clusters"] = df.apply(_decidir, axis=1)

tabla = df.pivot(index="word_normalized", columns="modelo", values="presenta_multiples_clusters")[MODELOS]

tasas_base = tabla.mean(skipna=True)
print("=== Tasa base de estructura multi-cluster por modelo (no es tasa de polisemia validada) ===")
print(tasas_base.to_string(), "\n")

tabla_completa = tabla.dropna()
n_final = len(tabla_completa)
print(f"Palabras evaluadas en los 5 modelos a la vez (excluye 'no determinado'/'no evaluada'): {n_final}")
print(f"n mínimo recomendado (sección 6 del protocolo, 10 comparaciones): {int(np.ceil(1.96**2 * 0.8*0.2 / (0.05**2 * 0.5**2)))} aprox. con alpha sin corregir; con Bonferroni ~2017\n")

# --- Cohen's Kappa pareado + Jaccard + bootstrap CI ---
filas_kappa = []
for m1, m2 in combinations(MODELOS, 2):
    y1 = tabla_completa[m1].astype(bool).values
    y2 = tabla_completa[m2].astype(bool).values
    kappa = cohen_kappa_score(y1, y2)
    ic_low, ic_high = bootstrap_kappa_ci(y1, y2)
    union = np.sum(y1 | y2)
    jaccard = np.sum(y1 & y2) / union if union > 0 else np.nan
    filas_kappa.append({
        "modelo_1": m1, "modelo_2": m2, "cohen_kappa": kappa,
        "ic_995_low": ic_low, "ic_995_high": ic_high, "jaccard": jaccard,
    })
kappa_df = pd.DataFrame(filas_kappa)
print("=== Cohen's Kappa pareado (10 combinaciones, IC 99.5% por Bonferroni) ===")
print(kappa_df.to_string(index=False), "\n")

# --- Fleiss' Kappa global ---
fleiss = fleiss_kappa(tabla_completa.astype(bool))
print(f"=== Fleiss' Kappa global (5 modelos, n={n_final}) ===\n{fleiss:.4f}\n")

# --- Índice de consenso por modelo ---
consenso = {m: [] for m in MODELOS}
for _, row in kappa_df.iterrows():
    consenso[row["modelo_1"]].append(row["cohen_kappa"])
    consenso[row["modelo_2"]].append(row["cohen_kappa"])
consenso_series = pd.Series({m: np.mean(v) for m, v in consenso.items()}).sort_values(ascending=False)
print("=== Índice de consenso por modelo (promedio de Kappa contra los otros 4) ===")
print(consenso_series.to_string(), "\n")

# --- Spearman en k_clusters, sobre palabras de consenso (>=3/5 con estructura multi-cluster) ---
tabla_k = df.pivot(index="word_normalized", columns="modelo", values="k_clusters")[MODELOS]
es_consenso = tabla_completa.sum(axis=1) >= UMBRAL_CONSENSO
palabras_consenso = tabla_completa[es_consenso].index
tabla_k_consenso = tabla_k.loc[palabras_consenso].dropna()
print(f"Palabras de consenso (>={UMBRAL_CONSENSO}/5 modelos coinciden en estructura multi-cluster), con k válido en los 5: {len(tabla_k_consenso)}")
print("(recomendado en protocolo: >=40-50 para esta parte)\n")

filas_spearman = []
for m1, m2 in combinations(MODELOS, 2):
    if len(tabla_k_consenso) >= 3:
        rho, p = spearmanr(tabla_k_consenso[m1], tabla_k_consenso[m2])
    else:
        rho, p = np.nan, np.nan
    filas_spearman.append({"modelo_1": m1, "modelo_2": m2, "spearman_rho": rho, "p_value": p})
spearman_df = pd.DataFrame(filas_spearman)
print("=== Correlación de Spearman en número de clusters (k), palabras de consenso ===")
print(spearman_df.to_string(index=False), "\n")

# --- Silhouette como medida descriptiva adicional (NO como clasificador -- ver
# protocolo: Youden J=0.215, señal real pero débil, no alcanza para gatear decisiones) ---
silhouette_por_modelo = df.groupby("modelo")["silhouette"].describe()
print("=== Distribución de silhouette por modelo (medida descriptiva, no umbral de decisión) ===")
print(silhouette_por_modelo.to_string(), "\n")

# --- Estabilidad multi-semilla como medida de calidad adicional (ver protocolo:
# reproducibilidad con 1 semilla no implica estabilidad -- "elefante" era 100%
# estable y aun así incorrecto, "internet"/"celular" muy inestables) ---
if "pct_semillas_moda" in df.columns:
    estabilidad_por_modelo = df.groupby("modelo")["pct_semillas_moda"].describe()
    print("=== Distribución de estabilidad multi-semilla (% de semillas que coinciden con la moda) ===")
    print(estabilidad_por_modelo.to_string(), "\n")
    estabilidad_por_modelo.to_csv("concordancia_estabilidad_descriptivo.csv")

# --- Descriptiva de k_clusters por modelo (NO depende del umbral de silhouette
# ni del gold standard de las 16 palabras -- es el conteo crudo de clusters) ---
tabla_k_todas = df.pivot(index="word_normalized", columns="modelo", values="k_clusters")[MODELOS]
print("=== Distribución de k_clusters por modelo (todas las palabras evaluadas, sin umbral) ===")
print(tabla_k_todas.describe().to_string(), "\n")

# --- Palabras con mayor desacuerdo entre modelos en k_clusters -- candidatas
# para revisión manual (¿los clusters "extra" de un modelo son sentidos reales,
# o solo ruido/fragmentación? Solo revisando caso por caso se puede afirmar
# que más clusters = mejor comprensión del idioma, nunca del conteo solo) ---
tabla_k_completa = tabla_k_todas.dropna()
desacuerdo = (tabla_k_completa.max(axis=1) - tabla_k_completa.min(axis=1))
top_desacuerdo = tabla_k_completa.loc[desacuerdo.sort_values(ascending=False).head(20).index]
top_desacuerdo["desacuerdo_max_menos_min"] = desacuerdo.loc[top_desacuerdo.index]
print("=== Top 20 palabras con mayor desacuerdo entre modelos en k_clusters (candidatas a revisión manual) ===")
print(top_desacuerdo.to_string())
top_desacuerdo.to_csv("concordancia_top_desacuerdo.csv")
print("\nGuardado: concordancia_top_desacuerdo.csv -- usa inspeccionar_ocurrencias() en clusterizar_sentidos.py")
print("sobre estas palabras para leer los tweets de cada cluster y confirmar si son sentidos distintos reales.\n")

# Guardar todo
kappa_df.to_csv("concordancia_kappa_pareado.csv", index=False)
consenso_series.to_csv("concordancia_indice_consenso.csv")
spearman_df.to_csv("concordancia_spearman_k.csv", index=False)
tasas_base.to_csv("concordancia_tasas_base.csv")
silhouette_por_modelo.to_csv("concordancia_silhouette_descriptivo.csv")
tabla_k_todas.describe().to_csv("concordancia_k_clusters_descriptivo.csv")
print("Guardado: concordancia_kappa_pareado.csv, concordancia_indice_consenso.csv, "
      "concordancia_spearman_k.csv, concordancia_tasas_base.csv, concordancia_silhouette_descriptivo.csv, "
      "concordancia_k_clusters_descriptivo.csv, concordancia_top_desacuerdo.csv")


=== Tasa base de estructura multi-cluster por modelo (no es tasa de polisemia validada) ===
modelo
mex_state     0.294737
mex_large     0.356271
beto          0.119845
bertin        0.250619
robertuito    0.303299 

Palabras evaluadas en los 5 modelos a la vez (excluye 'no determinado'/'no evaluada'): 3497
n mínimo recomendado (sección 6 del protocolo, 10 comparaciones): 984 aprox. con alpha sin corregir; con Bonferroni ~2017

=== Cohen's Kappa pareado (10 combinaciones, IC 99.5% por Bonferroni) ===
 modelo_1   modelo_2  cohen_kappa  ic_995_low  ic_995_high  jaccard
mex_state  mex_large     0.651551    0.613484     0.684824 0.650718
mex_state       beto     0.275991    0.236510     0.318748 0.272277
mex_state     bertin     0.190891    0.140719     0.239603 0.307443
mex_state robertuito     0.308219    0.259485     0.359247 0.394540
mex_large       beto     0.277999    0.240870     0.311651 0.278278
mex_large     bertin     0.216048    0.162958     0.260081 0.337084
mex_large robertuit

In [39]:
t2 = tabla_k_completa[tabla_k_completa['mex_state']>3]
acuerdo = (t2.max(axis=1) - t2.min(axis=1))

In [40]:
acuerdo.sort_values(ascending=True).head(20)

word_normalized
acabo        0.0
youtube      1.0
urge         1.0
bravo        1.0
recuerdos    1.0
personal     1.0
clase        1.0
ciudad       1.0
conoce       1.0
cuarta       1.0
paro         1.0
gusto        1.0
leche        1.0
home         1.0
deseo        1.0
municipio    1.0
federal      1.0
frente       1.0
vuelta       1.0
aire         1.0
dtype: float64

In [36]:
df[df['word_normalized']=='populista']

,word_normalized,modelo,n_ocurrencias,min_cluster_size_usado,n_componentes_quitar,k_clusters,pct_ruido,silhouette,pct_semillas_moda,presenta_multiples_clusters,excluida_stopword
9142,populista,mex_state,29,5.0,0.0,2.0,0.0,0.959899,100.0,True,False
21576,populista,mex_large,29,5.0,0.0,2.0,0.0,0.957387,100.0,True,False
34010,populista,beto,28,5.0,0.0,2.0,0.0,0.963492,100.0,True,False
46444,populista,bertin,29,5.0,0.0,2.0,0.0,0.959257,100.0,True,False
58878,populista,robertuito,29,5.0,0.0,2.0,0.0,0.948916,100.0,True,False


# Revisar palabras con desacuerdo

In [12]:
import os 
from collections import defaultdict
import umap
import hdbscan
from sklearn.metrics import silhouette_score
import contextlib

CARPETA_EMBEDDINGS = "embeddings"
# %%
MCS_FRACCION = 0.05   # min_cluster_size ≈ 5% de las ocurrencias de la palabra
MCS_MINIMO = 5
MCS_MAXIMO = 50

UMAP_N_COMPONENTS = 12
HDBSCAN_MIN_SAMPLES = 3
MIN_OCURRENCIAS_PARA_CLUSTER = 8

def cargar_embeddings(nombre_modelo, carpeta=CARPETA_EMBEDDINGS):
    ruta = os.path.join(carpeta, f"{nombre_modelo}_embeddings.npz")
    data = np.load(ruta, allow_pickle=True)
    emb = data["embeddings"]
    words = data["words"]
    id_tweets = data["id_tweets"]
    return emb, words, id_tweets

def calcular_min_cluster_size(n, fraccion=MCS_FRACCION, minimo=MCS_MINIMO, maximo=MCS_MAXIMO):
    return int(np.clip(round(fraccion * n), minimo, maximo))
    
def inspeccionar_ocurrencias(palabra, embeddings_dict, tweets_dict, min_cluster_size=None, min_samples=None, n_componentes_quitar=0):
    """Devuelve un DataFrame (id_tweet, cluster_id) para revisar caso por caso
    a qué tweet corresponde cada ocurrencia y en qué cluster cayó (cluster,
    no "sentido" — ver limitación en el encabezado del notebook)."""
    embs = embeddings_dict[palabra]
    tids = tweets_dict[palabra]
    mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(len(embs))
    ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES
    labels, _ = _hdbscan_labels(embs, mcs, ms, n_componentes_quitar=n_componentes_quitar)
    return pd.DataFrame({"id_tweet": tids, "cluster_id": labels})

def _hdbscan_labels(embeddings, min_cluster_size, min_samples, n_componentes_quitar=0):
    """Corre UMAP+HDBSCAN sobre los embeddings de una palabra y devuelve
    (labels, silhouette). labels: array de labels (uno por ocurrencia, -1 =
    ruido). silhouette: score de separación entre clusters (None si k<2 o muy
    pocos puntos no-ruido) -- útil para distinguir splits reales de espurios,
    algo que el número de clusters solo no puede decirnos. Si
    n_componentes_quitar>0, primero centra y quita esa(s) componente(s)
    principal(es) dominante(s) (ver quitar_componentes_dominantes)."""
    X = np.vstack(embeddings)
    if n_componentes_quitar > 0:
        X = quitar_componentes_dominantes(X, n_componentes_quitar)
    n = len(embeddings)
    n_components = min(UMAP_N_COMPONENTS, n - 2)
    n_neighbors = min(15, n - 2)
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_neighbors, random_state=0, n_jobs=1)
    X_red = reducer.fit_transform(X)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples)
    labels = clusterer.fit_predict(X_red)

    k = len(set(labels) - {-1})
    mask = labels != -1
    if k >= 2 and mask.sum() >= k + 1:
        silhouette = silhouette_score(X_red[mask], labels[mask])
    else:
        silhouette = None
    return labels, silhouette

def construir_dicts_para_palabras(nombre_modelo, palabras):
    emb, w, id_tweets = cargar_embeddings(nombre_modelo)
    palabras_arr = np.array([d for d in w])
    tweets_arr = np.array([d for d in id_tweets])
    emb_dict, tw_dict = defaultdict(list), defaultdict(list)
    for palabra in palabras:
        idx = np.where(palabras_arr == palabra)[0]
        if len(idx) == 0:
            print(f"'{palabra}' no está en el .npz de {nombre_modelo}")
            continue
        emb_dict[palabra] = list(emb[idx])
        tw_dict[palabra] = list(tweets_arr[idx])
    return emb_dict, tw_dict

In [15]:
"jaja,chido,juan,verde,busca,lucha,esperar,entiendo,temporada,rico,saltillo,leer,completa,sacar,vida,miedo,plan,gustan,pendeja,confianza"

'jaja,chido,juan,verde,busca,lucha,esperar,entiendo,temporada,rico,saltillo,leer,completa,sacar,vida,miedo,plan,gustan,pendeja,confianza'

In [41]:
MODELOS = ["mex_state", "mex_large", "beto", "bertin", "robertuito"]
#palabras_a_revisar = {"chido", "verde", "busca", "esperar", "entiendo", "leer", "vida", "confianza"} # 2
#palabras_a_revisar = {"lucha", "rico", "temporada", "saltillo", "juan"} # 1
palabras_a_revisar = {"populista", "homenaje", "carita", "acabo"} # 3
textos_df = pd.read_csv("output/all_tweets.csv") 
for nombre_modelo in MODELOS:
    
    
    with open(f"output/revision_palabras3_{nombre_modelo}.txt", "w", encoding="utf-8") as f, contextlib.redirect_stdout(f):
        emb_dict_beto, tw_dict_beto = construir_dicts_para_palabras(nombre_modelo, palabras_a_revisar)
        for p in palabras_a_revisar:
            print("-"*50)
            print("Palabra:", p)
            tabla = inspeccionar_ocurrencias(p, emb_dict_beto, tw_dict_beto)
            tabla_con_texto = tabla.merge(textos_df, on="id_tweet", how="left")
            for cluster_id, grupo in tabla_con_texto.groupby("cluster_id"):
                print(f"\n--- cluster {cluster_id} ({len(grupo)} ocurrencias) ---")
                lista = grupo["texto"].sample(min(5, len(grupo)), random_state=42).to_list()
                for l in lista:
                    print(l)
                    print()
    
    print(f"Listo, revisa revision_palabras3_{nombre_modelo}.txt")


Listo, revisa revision_palabras3_mex_state.txt
Listo, revisa revision_palabras3_mex_large.txt
Listo, revisa revision_palabras3_beto.txt
Listo, revisa revision_palabras3_bertin.txt
Listo, revisa revision_palabras3_robertuito.txt
